# P2 — Saved locked-test review
Retrospective research. These cells read saved artifacts; they do not rerun model selection or the holdout. Run from the repository root or notebooks directory. Private input prices are needed only to reproduce the earlier training pipeline.


In [1]:
from pathlib import Path
import json
r = Path.cwd()
if not (r/'artifacts/research').exists(): r=r.parent
b=json.loads((r/'artifacts/research/public_bundle.json').read_text())
print(b['status'], b['as_of'], b['selected'])

REAL DATA · RETROSPECTIVE EXPLORATORY 2025-12-31 stumps_24


In [2]:
for name, result in b['results'].items():
 m=result['metrics']
 print(name, 'CAGR', round(100*m['cagr'],2), 'Sharpe',round(m['sharpe_zero_cash'],3), 'MDD',round(100*m['max_drawdown'],2))

equal_weight CAGR 14.16 Sharpe 1.088 MDD -15.59
momentum_63 CAGR 13.19 Sharpe 0.936 MDD -16.63
inverse_vol CAGR 12.37 Sharpe 1.006 MDD -15.3
stumps_24 CAGR 12.41 Sharpe 0.785 MDD -19.94
SPY_buy_hold CAGR 22.39 Sharpe 1.418 MDD -18.31


## Development selection
All six candidates remain visible. Only the development-selected candidate enters the saved test comparison.

In [3]:
for name,m in b['development_metrics'].items(): print(name,round(m['sharpe_zero_cash'],4),name==b['selected'])

equal_weight 0.5233 False
momentum_63 0.6456 False
inverse_vol 0.5421 False
ridge_0_1 0.3148 False
ridge_10 0.3148 False
ridge_100 0.3259 False
enet_0_001 0.2947 False
enet_0_01 0.2585 False
stumps_24 0.462 True
SPY_buy_hold 0.4103 False


In [4]:
print(json.dumps(b['uncertainty'],indent=2))
print(b['conclusion'])

{
  "paired_months": 36,
  "mean_monthly_net_difference": -0.00031472322707744153,
  "ci95_low": -0.009998581995489303,
  "ci95_high": 0.009885648664416588,
  "block_months": 3,
  "repetitions": 2000,
  "seed": 20261124,
  "interpretation": "Circular time-block bootstrap interval for arithmetic mean paired monthly return difference; descriptive small-sample uncertainty, not proof of alpha."
}
The selected ML model did not outperform momentum or equal-weight in the locked sample. The paired interval includes zero; no established incremental edge and no trading promotion.


In [5]:
assert all(x['max_label_end'] < x['signal_date'] < x['fill_date'] for x in b['fits'])
print('Temporal fit windows checked:',len(b['fits']))
for n,result in b['results'].items():
 residual=max(abs(x['nav']-x['cash']-x['market_value']-x['dividend_receivable']) for x in result['ledger'])
 assert residual < 1e-7
 print(n, 'NAV residual', residual)

Temporal fit windows checked: 36
equal_weight NAV residual 2.3646862246096134e-11
momentum_63 NAV residual 1.4551915228366852e-11
inverse_vol NAV residual 1.4551915228366852e-11
stumps_24 NAV residual 1.8417267710901797e-11
SPY_buy_hold NAV residual 1.8417267710901797e-11


In [6]:
for name,m in b['stress'].items(): print(name, round(100*m['cagr'],2),round(m['sharpe_zero_cash'],3))

stumps_24__cost0 13.07 0.821
stumps_24__cost1 12.41 0.785
stumps_24__cost2 11.77 0.75
stumps_24__cost3 11.11 0.714
stumps_24__delay1 15.83 0.976
momentum_63__cost0 13.87 0.978
momentum_63__cost1 13.19 0.936
momentum_63__cost2 12.51 0.893
momentum_63__cost3 11.83 0.85
momentum_63__delay1 11.54 0.83
equal_weight__cost0 14.2 1.092
equal_weight__cost1 14.16 1.088
equal_weight__cost2 14.11 1.085
equal_weight__cost3 14.06 1.081
equal_weight__delay1 14.47 1.111


## First rebalance
The workbook provides independent formulas for a cash-only initial rebalance. It does not replicate the complete walk-forward model or all corporate-action accounting.

In [7]:
v=b['review'];cash=v['initial_nav'];market=0
for a in v['assets']:
 px=a['raw_open']*(1+v['slippage_bps']/10000)
 q=min(int(v['initial_nav']*a['target_weight']/a['raw_open']),int(a['volume']*v['participation']),int(max(0,cash)/(px*(1+v['fee_bps']/10000))))
 cash-=q*px*(1+v['fee_bps']/10000);market+=q*a['raw_close']
 print(a['asset'],q)
assert abs(cash+market-v['python_ledger']['nav'])<1e-7
print('NAV:',cash+market,'cash:',cash)

XLB 0
XLE 383
XLF 0
XLI 0
XLK 265
XLP 0
XLU 0
XLV 0
XLY 256
NAV: 97918.169238186 cash: 63.789238186007424


In [8]:
print(json.dumps(b['paper'],indent=2))
print('No prospective track record or current trading signal.')

{
  "status": "BLOCKED_STALE_DATA",
  "realised_forward_sessions": 0,
  "signals_published": 0,
  "message": "Historical snapshot ends2025-12-31. No current trade signal or prospective performance is available."
}
No prospective track record or current trading signal.
